# Von der Theorie zur Praxis: Geo-KI Analysen selber durchführen

Michael Mommert (michael.mommert@hft-stuttgart.de), Volker Coors, HFT Stuttgart, 2026

In diesem Praxisbeispiel werden wir vortrainierte UNet-Modelle auf verschiedene Bilddaten anwenden um unterschiedliche Vegetationsformen zu erkennen. Wir verwenden hierfür sowohl Satellitendaten ([Sentinel-2](https://de.wikipedia.org/wiki/Sentinel-2)) mit 10m Auflösung, also auch Digitale Orthophotos mit 20cm Auflösung am Boden (DOP20). Das Ziel ist es, die Ausgabe des jeweiligen Modells als georeferenzierte GeoTIFF-Datei abzuspeichern, um dieses Produkt dann später weiterverwenden zu können.

Die in dieser Übung verwendeten Daten stammen aus folgenden Quellen:
* Satellitenbilder: Sentinel-2, Copernicus Programm, Europäische Kommission
* Orthophotos: LGL, [www.lgl-bw.de](www.lgl-bw.de), dl-de/by-2-0

## Software und Entwicklungsumgebung

Für diese Übung benutzen wir ausschliesslich Open-Source Software. Das Training und die Auswertung der Modelle erfolgen in der Programmiersprache Python, welche sehr gut für den Umgang mit unterschiedlichen Datentypen und die Erstellung von KI-Anwendungen geeignet ist. Unsere KI-Modelle wurden basierend auf PyTorch erstellt, welche wir auch im Folgenden für die Auswertung benutzen werden.

Die Umgebung, welche Sie hierfür benutzen, ist ein Jupyter Notebook. Dieses besteht aus Zellen. Man unterscheidet Textzellen, wie diese hier, und Code-Zellen, welche ausführbaren Code enthalten. Zum Ausführen des Codes klicken Sie einfach auf das kleine Play-Symbole, oder die klicken in die Zelle und drücken die Tastenkombination `Umstelltaste` + `Enter`. 

## Download der Daten und Hilfsfunktionen

Bevor wir mit der Übung beginnen, müssen wir die vorbereiten Daten und ein paar Hilfsfunktionen herunterladen und entpackn (dies kann einen Moment dauern):

In [ ]:
!curl -o 3dForumLindau.zip -L https://bwsyncandshare.kit.edu/public.php/dav/files/CF3A5EHf3Bp5qBt/?accept=zip
!unzip 3dForumLindau.zip

Wenn alles heruntergeladen wurde, importieren wir noch ein paar notwendige Module und unsere Hilfsfunktionen:

In [ ]:
# import modules
import matplotlib.pyplot as plt
import rasterio as rio
import torch

# import helpers
from unet import UNet
from data import Scene

## Auswertung von Satellitenbilddaten

Im ersten Teil der Übung laden wir nun Bilddaten des Sentinel-2 Satelliten ein. Die Daten liegen im Verzeichnis `data/sen2`; unterschiedliche Szenen wurden vorbereitet:
* Lindau, aufgenommen am 19. Juni 2025, relativer Pfad: `data/lindau/sen2_lindau_20250619.tif`
* Stuttgart, aufgenommen am 30. April 2026, relativer Pfad: `data/stuttgart/sen2_stuttgart_20260430.tif`
* Rom, aufgenommen am 3. Juni 2026, relativer Pfad: `data/rom/sen2_rom_20250603.tif`

Die Daten wurden über OpenEO heruntergeladen, welches eine direkte Schnittstelle zum Copernicus Archiv bietet. Die Bilder enthalten die Bänder 2 (Blau), 3 (Grün), 4 (Rot) und 8 (Nahinfrarot) jeweils mit einer Auflösung von 10m am Boden.

Wir beginnen mit dem Lindau-Bilder und laden dieses über die Hilfsfunktion:

In [ ]:
scene_filename = 'data/lindau/sen2_lindau_20250619.tif'
scene = Scene(scene_filename)

Wir können das Bild über eine `display()`-Methode anzeigen lassen:

In [ ]:
scene.display()

Wir möchten nun ein vortrainiertes UNet-Modell auf dieses Bild anwenden, um unterschiedliche Landbedeckungstypen zu erkennen. Das Modell wurde auf Sentinel-2-Daten und Landbeckung nach dem ESA WorldCover-Schema trainiert.

Wir instanzieren das Modell und laden dann die vortrainierten Gewichte (das Wissen) in das Modell ein:

In [ ]:
# instantiate model
model = UNet(n_channels=4, n_classes=12)

# read in model weights
model_path = "models/unet_sen2_esaworldcover.pth"
model.load_state_dict(torch.load(model_path, weights_only=True, map_location=torch.device('cpu')))
model.eval()

Jetzt können wir das eingeladene Satellitenbild durch das Modell schicken, um die Klassifizierung durchzuführen, d.h. jedem Pixel des Bildes eine Klasse zuzuordnen. 

Dazu muss zunächst der Datentyp und die Darstellung der Bilddaten angepasst werden. Die Ausgabe des Modells sind einfache Aktivierungen über alle möglichen 12 Klassen. Für jeden Pixel übernehmen wir die Klasse, welche die höchsten Aktivierungswerte hat. Das resultierende Array `pred` enthält die entsprechende Klasse jedes einzelnen Bildpixels. 

In [ ]:
# reshaping the image array and recasting it as a tensor
x = torch.unsqueeze(torch.Tensor(scene.data), dim=0)

# deactivate computational graph
with torch.no_grad():

    # run tensor through the model
    output = model(x)[0]

    # extract predictions using argmax
    pred = torch.argmax(output, dim=0).numpy().astype('uint8')


Wir visualisieren die Vorhersagen des Modells im Vergleich zum Originalbild.

In [ ]:
scene.display(pred=pred)

Die Ergebnisse sehen gut aus. Für mehr Details wählen wir einen Ausschnitt aus:

In [ ]:
scene.display(pred=pred, aoi=(200, 500, 350, 400))

Die Detailansicht zeigt einzelne Fehler (z.B. im Hafenbereich). Im Grossen und Ganzen ist das Ergebnis aber überzeugend.

Zum Abschluss speichern wir die resultierende Karte noch als georeferenzierte GeoTIFF-Datei ab. Da die Ausgabe deckungsgleich mit dem Satellitenbild ist, können wir einfach die Georeferenzierung von dort übernehmen.

In [ ]:
with rio.open(
    scene_filename.replace('.tif', '_lulc.tif'), 
    'w',
    driver='GTiff',
    height=pred.shape[0],
    width=pred.shape[1],
    count=1,
    dtype=pred.dtype,
    crs=scene.crs,
    transform=scene.transform,
) as dst:
    dst.write(pred, 1)

Die Datei liegt im selben Verzeichnis, wie die Bilddaten. Von dort kann sie beispielweise in ein GIS-Tool geladen werden, oder anderweitig weiterverwendet werden.

Die anderen Satellitenbilder können ebenso ausgewertet werden. Hierzu müssen einfach der Dateiname `scene_filename` und die Koordinaten des Ausschnitts angepasst werden.

## Auswertung von Orthophotos

Im zweiten Teil der Übung verwenden wir ein anderes vortrainiertes Modell zur Extraktion von hoher und niedriger Vegetation aus Orthophotos. Hierfür steht uns drei Beispielbilder im Ordner `data/dop20/` zur Verfügung, welche unterschiedliche Gebiete abdecken:
* `dop20_vorort.tif` zeigt einen vorstädtischen Bereich,
* `dop20_stadt.tif` zeigt einen städtischen Bereich und 
* `dop20_landwirtschaft.tif` zeigt einen landwirtschaftlich genutzten Bereich.

Wir beginnen mit dem Ausschnitt aus einem Vorort und zeigen diesen an:

In [ ]:
scene_filename = 'data/dop20/dop20_vorort.tif'
scene = Scene(scene_filename)

scene.display()

Wie zuvor bei den Satellitendaten laden wir nun das trainierte Modell in den Speicher...

In [ ]:
# instantiate model
model = UNet(n_channels=4, n_classes=3)

# read in model weights
model_path = "models/unet_dop_veg.pth"
model.load_state_dict(torch.load(model_path, weights_only=True, map_location=torch.device('cpu')))
model.eval()

... und schicken das Bild durch das neue Modell.

In [ ]:
# reshaping the image array and recasting it as a tensor
x = torch.unsqueeze(torch.Tensor(scene.data), dim=0)

# deactivate computational graph
with torch.no_grad():

    # run tensor through the model
    output = model(x)[0]

    # extract predictions using argmax
    pred = torch.argmax(output, dim=0).numpy().astype('uint8')

Wir schauen uns das Ergebnis an:

In [ ]:
scene.display(pred=pred)

Wir sehen, dass das Modell sehrwohl in der Lage ist, hohe Vegetation (orange) und flache Vegetation (gelb) zu unterscheiden. Es zeigen sich aber auch Probleme: beispielsweise hat das Modell Probleme damit, Vegetation in verschatteten Bereichen zu erkennen. Auch ist die Fähigkeit des Modells, sehr kleinteilige begrünte Flächen zu erkennen, eingeschränkt.

Auch dieses Ergebnis schreiben wir in eine GeoTIFF-Datei, damit wir damit weiterarbeiten können.

In [ ]:
with rio.open(
    scene_filename.replace('.tif', '_lulc.tif'), 
    'w',
    driver='GTiff',
    height=pred.shape[0],
    width=pred.shape[1],
    count=1,
    dtype=pred.dtype,
    crs=scene.crs,
    transform=scene.transform,
) as dst:
    dst.write(pred, 1)

## Zusammenfassung

* Es ist relativ einfach, ein fertig trainiertes Modell auf unterschiedlichen Bilddaten auszuwerten. 
* Für kleine Datenmengen werden hierfür keine GPUs benötigt.
* Die Qualität der Ergebnisse hängt sowohl von der Qualität des trainierten Modells, aber auch von den Bilddaten ab.